In [ ]:
# === STANDART COLAB KURULUM HÜCRESİ (her notebook'un ilk hücresi) ===
from google.colab import drive, userdata
drive.mount("/content/drive")

import os, sys
REPO_URL = "https://github.com/Oldzhay34/receipts-and-interactions.git"
BRANCH   = "a/1.4-outliers"                                  # merge edildikten sonra "main"
try:
    TOKEN = userdata.get("GH_TOKEN")                        # sadece private repo için gerekir
except userdata.SecretNotFoundError:
    TOKEN = None
url = REPO_URL.replace("https://", f"https://{TOKEN}@") if TOKEN else REPO_URL

if not os.path.exists("/content/repo"):
    !git clone -q -b {BRANCH} {url} /content/repo
else:
    !cd /content/repo && git fetch -q && git checkout -q {BRANCH} && git pull -q

%cd /content/repo
!pip install -q -r requirements.txt && pip install -q -e .
if "/content/repo/src" not in sys.path:      # editable install is only seen after a kernel restart
    sys.path.insert(0, "/content/repo/src")

os.environ["DATA_ROOT"] = "/content/drive/MyDrive/CME4403/data"
from recsys.config import load_config
CFG = load_config()          # config/config.yaml + DATA_ROOT
print("commit:", os.popen("git rev-parse --short HEAD").read().strip())

# 04 — Outliers

Stage 1.4. This notebook contains no business logic; it only calls `recsys.cleaning.outliers`
(and `ZeroAsMissing` from 1.3, so that `minutes = 0` is not mistaken for a small value).

* Inputs: stages **`loaded`** and **`quality`** (LATEST), validated against their schemas.
* Output: one **`outliers`** version with the evidence tables below (each validated against
  `OUTLIERS_OUTPUT_SCHEMA`).
* Columns: `minutes`, `n_steps`, `n_ingredients` and the 7 nutrition columns.
* The analysis uses the full data because it is exploratory. **The transformers are only
  demonstrated here: their real fit happens in the 1.9 pipeline, on train data only.**
* The figures use plain matplotlib; the shared figure style comes in 1.6.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from recsys.cleaning.missing import ZeroAsMissing
from recsys.cleaning.outliers import (
    OUTLIER_COLUMNS,
    before_after_table,
    extreme_examples,
    invalid_counts,
    iqr_table,
    outlier_steps_from_config,
    quantile_table,
    threshold_counts,
)
from recsys.contracts import (
    LOADED_INTERACTIONS_SCHEMA,
    LOADED_RECIPES_SCHEMA,
    OUTLIERS_OUTPUT_SCHEMA,
    QUALITY_ISSUES_SCHEMA,
    RECIPES_AFTER_OUTLIERS_SCHEMA,
    validate,
)
from recsys.io import ArtifactStore

OUT_CFG = CFG.section("outliers")    # config.yaml `outliers` block (strategies per column)
COLS = list(OUTLIER_COLUMNS)
N_EXAMPLES = 20
# candidate upper bounds for minutes: 1, 2, 7, 14, 30, 60 days and 1 year
MINUTES_THRESHOLDS = (1440, 2880, 10080, 20160, 43200, 86400, 525600)
RELATED_ISSUES = ("minutes_zero", "minutes_extreme", "n_steps_zero")

store = ArtifactStore.from_config(CFG)
loaded = store.load("loaded")
quality = store.load("quality")
recipes = validate(loaded["recipes"], LOADED_RECIPES_SCHEMA)
validate(loaded["interactions"], LOADED_INTERACTIONS_SCHEMA)
issues = validate(quality["issues"], QUALITY_ISSUES_SCHEMA)
related = issues[issues["issue_id"].isin(RELATED_ISSUES)].reset_index(drop=True)
print("inputs: loaded", loaded.version, "| quality", quality.version)
print("recipes", recipes.shape)
related

## 1. IQR outliers

Tukey fences `Q1 - 1.5 IQR` / `Q3 + 1.5 IQR` on the raw scale and on the `log1p` scale. All
columns are strongly right-skewed, so the raw-scale fences flag many ordinary recipes; the
log-scale fences flag far fewer. `minutes = 0` is treated as missing (1.3) before counting.

In [ ]:
recipes_z = ZeroAsMissing(columns=("minutes",)).fit_transform(recipes)
iqr = pd.concat([iqr_table(recipes_z, COLS), iqr_table(recipes_z, COLS, log1p=True)],
                ignore_index=True)
iqr[["column", "scale", "n", "q1", "median", "q3", "lower_fence", "upper_fence",
     "n_below", "n_above", "share_outside", "max"]]

In [ ]:
# Box plots from the IQR table (raw scale, outliers hidden: their counts are in the title).
raw = iqr[iqr["scale"] == "raw"].reset_index(drop=True)
fig, axes = plt.subplots(2, 5, figsize=(18, 7))
for ax, r in zip(axes.flat, raw.itertuples()):
    ax.bxp([{"label": r.column, "med": r.median, "q1": r.q1, "q3": r.q3,
             "whislo": r.whisker_low, "whishi": r.whisker_high, "fliers": []}],
           showfliers=False)
    ax.set_title(f"{r.column}\n{r.n_above} above fence, max {r.max:,.0f}", fontsize=9)
fig.suptitle("Box plots (raw scale, Tukey 1.5 IQR)")
fig.tight_layout()
plt.show()

## 2. Invalid values vs real extremes

* **(a) Invalid / impossible**: a value that cannot describe a recipe (a preparation time of
  months or years, a recipe with 0 steps or 0 ingredients, negative nutrition). These become NaN
  (`InvalidValueRule`); rows are not dropped.
* **(b) Real but extreme**: a value that is possible but rare (very caloric cakes, long
  marinades, 20+ ingredient dishes). These are kept and only clipped at a train-learned quantile
  (`QuantileClipper`) and, where still very skewed, log-transformed (`Log1pTransformer`).

The thresholds are read from the quantiles, the counts above candidate bounds and the extreme
examples below.

In [ ]:
quantiles = quantile_table(recipes_z, COLS)
display(quantiles.pivot(index="column", columns="quantile", values="value"))
minutes_thr = threshold_counts(recipes_z, "minutes", MINUTES_THRESHOLDS)
minutes_thr

In [ ]:
examples = pd.concat(
    [extreme_examples(recipes_z, "minutes", n=N_EXAMPLES),
     extreme_examples(recipes_z, "calories", n=N_EXAMPLES)],
    ignore_index=True,
)
examples

In [ ]:
bounds = OUT_CFG["invalid_rule"]
invalid = invalid_counts(recipes_z, bounds)
invalid

## 3. Strategies from config (demo fit)

`outlier_steps_from_config` builds the transformers listed in the `outliers` block of
`config/config.yaml`. The clipper is fitted on a random share of the recipes, as a stand-in for
the train split; the real fit happens in 1.9 on train only. Values removed by the invalid rule
are NaN before the clip bounds are learned, so impossible values do not move the bounds.

In [ ]:
steps = dict(outlier_steps_from_config(OUT_CFG))
print("steps:", list(steps))
rule, clipper, log1p = steps["invalid_rule"], steps["quantile_clip"], steps["log1p"]

demo_fit = recipes_z.sample(frac=CFG.split["train"], random_state=CFG.random_state)
clipper.fit(rule.fit_transform(demo_fit))

after_rule = rule.transform(recipes_z)
after_clip = validate(clipper.transform(after_rule), RECIPES_AFTER_OUTLIERS_SCHEMA)
after_log = log1p.fit_transform(after_clip)
clip_bounds = clipper.bounds_table(after_rule)
clip_bounds

## 4. Before / after

`before` = loaded data (`minutes = 0` already NaN), `after_rules` = after the invalid rule and
the quantile clip (original scale), `after_log1p` = after the log transform.

In [ ]:
before_after = before_after_table(
    {"before": recipes_z, "after_rules": after_clip, "after_log1p": after_log}, COLS
)
before_after.pivot(index="column", columns="stage", values="skew")[
    ["before", "after_rules", "after_log1p"]]

In [ ]:
# Box plots before / after (outliers hidden; the table above has the counts).
fig, axes = plt.subplots(2, 5, figsize=(18, 7))
for ax, col in zip(axes.flat, COLS):
    part = before_after[before_after["column"] == col]
    ax.bxp([{"label": r.stage, "med": r.median, "q1": r.q1, "q3": r.q3,
             "whislo": r.whisker_low, "whishi": r.whisker_high, "fliers": []}
            for r in part.itertuples() if r.stage != "after_log1p"],
           showfliers=False)
    ax.set_title(col, fontsize=9)
fig.suptitle("Before / after invalid rule + quantile clip (original scale)")
fig.tight_layout()
plt.show()

fig, axes = plt.subplots(2, 5, figsize=(18, 7))
for ax, col in zip(axes.flat, COLS):
    ax.hist(after_log[col].dropna(), bins=50)
    ax.set_title(f"{col} after {'log1p' if col in log1p.columns else 'clip'}", fontsize=9)
fig.tight_layout()
plt.show()

## 5. Save stage `outliers`

In [ ]:
outliers = store.save(
    "outliers",
    {
        "iqr": iqr,
        "quantiles": quantiles,
        "minutes_thresholds": minutes_thr,
        "extreme_examples": examples,
        "invalid_counts": invalid,
        "clip_bounds": clip_bounds,
        "before_after": before_after,
        "related_issues": related,
    },
    inputs=[loaded, quality],
    params={
        "columns": COLS,
        "outliers_config": OUT_CFG,
        "minutes_thresholds": list(MINUTES_THRESHOLDS),
        "n_examples": N_EXAMPLES,
        "demo_fit_frac": CFG.split["train"],
        "random_state": CFG.random_state,
    },
    schemas=OUTLIERS_OUTPUT_SCHEMA,
)
print("saved:", outliers.stage, outliers.version, "->", outliers.path)
check = store.load("outliers")  # re-read with sha256 verification
{name: f["n_rows"] for name, f in check.manifest["files"].items()}